<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# 若在 Google Colab 上執行，請先安裝中文字型支援庫與下載工具
%pip install wget mplfonts


# 📈 範例 01：廣告效益預測 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本筆記本採用**「一段程式碼 + 一段 AI 互動學習指引」**的設計架構：
- 每一段程式碼都經過嚴格測試，保證可以直接執行。
- 執行完程式碼後，**下方的文字儲存格**會詳細解釋程式在做什麼，並提供專屬的 **Prompt（提示詞）**，讓你可以複製貼給 AI，請 AI 替你解說細節、深入探討觀念、或幫你修改程式碼！

---

## 📊 一、資料集介紹 (`advSale.csv`)

👉 **資料集直接下載**：[advSale.csv (點此下載或於新分頁開啟)](https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/advSale.csv)

本數據集記錄了某企業在 4,500 個行銷週期中，各廣告媒體的預算投入與最終產品銷售額：

| 欄位名稱 | 資料型態 | 單位 / 數值範例 | 業務意義 |
| :--- | :--- | :--- | :--- |
| **電視廣告** | 數值型 (`float64`) | 93.0 (千元) | 電視廣告費用 |
| **廣播廣告** | 數值型 (`float64`) | 25.6 (千元) | 廣播電台廣告費用 |
| **社群媒體廣告** | 數值型 (`float64`) | 5.1 (千元) | FB/IG 等網路社群廣告費用 |
| **網紅廣告** | 類別型 (`object`) | 「大於百萬」、「小於一萬」 | 合作網紅粉絲規模級距 |
| **銷售額 (目標 y)** | 數值型 (`float64`) | 334.5 (萬元) | **我們想預測的目標連續變數** |

---

## 🧭 二、機器學習全流程地圖

我們將帶領你走過專業資料科學家的標準 6 大步驟：
1. **環境配置與字型**：解決繪圖中文方塊亂碼問題。
2. **資料載入與探索 (EDA)**：檢查形狀、缺失值與資料前幾筆長相。
3. **相關性分析與視覺化**：繪製熱力圖，找出誰是帶動銷售的關鍵管道。
4. **特徵工程與 Pipeline 建置**：切分 80% 訓練與 20% 測試集，利用管線封裝防止資料洩漏。
5. **模型選型與交叉驗證比較**：比較「多元線性迴歸 (Baseline)」與「隨機森林迴歸」。
6. **測試集大考與業務預測**：解讀 R²、RMSE 與 MAE，並進行新預算方案試算。

---  
### 【步驟 0】：環境準備與繁體中文字型設定
在開始之前，我們先載入標準資料科學套件，並設定好繁體中文字型，確保後續繪圖不會出現豆腐塊亂碼 (`□□`)。

In [ ]:
# [步驟 0 程式碼] 載入套件與設定中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 自動設定繁體中文字型（支援 mplfonts 與各作業系統系統字型）
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Microsoft JhengHei", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas（資料處理）、NumPy（矩陣運算）、Matplotlib/Seaborn（資料視覺化），並利用 `mplfonts` 或系統字型設定中文字型支援，避免繪圖時中文變成亂碼方塊。

- **你可以複製問 AI 的 Prompt（深入理解）：**
  > 「請解釋為什麼 Matplotlib 在預設情況下繪製中文會出現亂碼或方塊？程式碼中的 `axes.unicode_minus = False` 又是在解決什麼問題？」

- **讓 AI 幫你修改這段程式的 Prompt（客製化）：**
  > 「我使用的是 Windows 電腦，如果我想指定使用『微軟正黑體 (Microsoft JhengHei)』，這段字型設定程式碼該如何簡化？」

---  
### 【步驟 1】：載入資料集與檢視基本結構
執行下方儲存格，讀入 `advSale.csv`，檢視總筆數、欄位資料型態，以及前 5 筆樣本數據。

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
import os

# 若本地不存在 advSale.csv（例如在 Google Colab 環境中執行），自動從 GitHub 下載
if not os.path.exists("advSale.csv"):
    import wget
    print("正在下載 advSale.csv 資料集...")
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/advSale.csv")
    print("\n✅ 下載完成！")

df = pd.read_csv("advSale.csv")

print("=== 資料結構與缺失值摘要 ===")
df.info()

print("\n=== 前 5 筆資料樣本 ===")
display(df.head())

print("\n=== 網紅廣告各級距分佈統計 ===")
print(df["網紅廣告"].value_counts())


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  `pd.read_csv` 將 CSV 讀成表格 DataFrame。`df.info()` 顯示共有 4,500 筆資料且無任何缺失值（Non-Null Count 全為 4500）。`value_counts()` 統計了「網紅廣告」中四種文字級距的分佈人數。

- **你可以複製問 AI 的 Prompt（看不懂時問）：**
  > 「這是我在 Jupyter Notebook 執行的資料資訊：共有 4500 筆資料，欄位有電視廣告、廣播廣告、社群媒體廣告、網紅廣告、銷售額。請用白話文告訴我，這份資料能拿來做什麼機器學習應用？目標 y 是哪一個？」

- **讓 AI 幫你修改這段程式的 Prompt（進階探索）：**
  > 「請幫我寫一行程式碼，用 Pandas 的 `.describe()` 算出數值欄位的平均值、標準差、最大值與最小值，並告訴我電視廣告的平均花費是多少？」

---  
### 【步驟 2】：特徵相關性分析與視覺化熱力圖 (EDA)
執行下方儲存格，計算各數值型廣告費用與銷售額之間的皮爾森相關係數 (Correlation)，並繪製相關性熱力圖。

In [ ]:
# [步驟 2 程式碼] 相關係數計算與熱力圖繪製
num_cols = ["電視廣告", "廣播廣告", "社群媒體廣告", "銷售額"]
corr = df[num_cols].corr()

print("=== 各廣告與銷售額之相關係數排序 ===")
print(corr["銷售額"].sort_values(ascending=False))

# 繪製熱力圖
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f", cbar=True)
plt.title("廣告媒體預算與銷售額之相關性熱力圖", fontsize=13)
plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  計算欄位兩兩之間的相關係數（介於 -1 到 +1 之間）。熱力圖顯示「電視廣告」與「銷售額」的係數高達 0.99，代表兩者幾乎呈現極強的線性正比關係！廣播廣告為 0.25，也有一定的關聯。

- **你可以複製問 AI 的 Prompt（解讀觀念）：**
  > 「我看到電視廣告與銷售額的相關係數高達 0.99，而社群媒體廣告只有 0.08。請告訴我：相關係數高代表什麼商業意涵？相關係數低是否就代表社群廣告完全無效？」

- **讓 AI 幫你修改這段程式的 Prompt（換圖表）：**
  > 「請幫我修改這段程式碼，改為繪製『電視廣告 vs. 銷售額』的二維散佈圖 (Scatter Plot)，並在上面加上一條紅色的最佳擬合趨勢線。」

---  
### 【步驟 3】：特徵工程與 Pipeline 建置 (嚴防資料洩漏)
執行下方儲存格，切分特徵 $X$ 與目標 $y$，切分訓練集 (80%) 與測試集 (20%)，並利用 `ColumnTransformer` 將數值標準化與類別獨熱編碼封裝。

In [ ]:
# [步驟 3 程式碼] 資料切分與特徵前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. 切分特徵 X 與目標 y
X = df.drop(columns=["銷售額"])
y = df["銷售額"]

# 2. 定義前處理器：數值欄位做 StandardScaler，文字欄位做 OneHotEncoder
numeric_features = ["電視廣告", "廣播廣告", "社群媒體廣告"]
categorical_features = ["網紅廣告"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_features)
    ]
)

# 3. 切分訓練集 (80%) 與測試集 (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ 資料切分完成！訓練集樣本數: {X_train.shape[0]}, 測試集樣本數: {X_test.shape[0]}")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們把資料切分成訓練集與測試集。更重要的是，我們定義了 `ColumnTransformer`，確保數值縮放與類別編碼規則「只會在訓練集上學習 (fit)」，封存測試集，達到機器學習工程最關鍵的「防資料洩漏 (Anti-Data Leakage)」。

- **你可以複製問 AI 的 Prompt（避坑觀念）：**
  > 「請用白話文向我這個新手解釋：為什麼我們不能在 `train_test_split` 之前，就對整份 DataFrame 做 `StandardScaler`？什麼叫做『資料洩漏 (Data Leakage)』？」

- **讓 AI 幫你修改這段程式的 Prompt（自訂比例）：**
  > 「我想把測試集改為 30%（即 70% 訓練、30% 測試），並且想保留所有 One-Hot 編碼欄位（不丟棄第一個基準值 `drop=first`），這段程式該如何調整？」

---  
### 【步驟 4】：模型選型與 5 折交叉驗證比較 (Baseline vs. 進階模型)
執行下方儲存格，建立「多元線性迴歸 (Baseline)」與「隨機森林迴歸 (Random Forest)」，並透過 5 折交叉驗證評估穩定度。

In [ ]:
# [步驟 4 程式碼] Pipeline 整合與交叉驗證評估
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 建立包含前處理與模型的 Pipeline
model_lr = Pipeline([("prep", preprocessor), ("regressor", LinearRegression())])
model_rf = Pipeline([("prep", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))])

# 執行 5 折交叉驗證 (以 R² 為評估指標)
scores_lr = cross_val_score(model_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(model_rf, X_train, y_train, cv=5, scoring="r2")

print(f"[多元線性迴歸] 5折交叉驗證平均 R²: {scores_lr.mean():.4f} (標準差: {scores_lr.std():.4f})")
print(f"[隨機森林迴歸] 5折交叉驗證平均 R²: {scores_rf.mean():.4f} (標準差: {scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們把前處理與演算法焊接成 `Pipeline`。利用「5 折交叉驗證」輪流將訓練集切成 5 等分進行訓練與考驗。多元線性迴歸與隨機森林的 R² 皆高達 0.998，代表線性關係非常顯著！

- **你可以複製問 AI 的 Prompt（模型選型觀念）：**
  > 「我看到多元線性迴歸與隨機森林的 R² 分數幾乎一樣高（約 0.998）。在實務商業應用上，既然簡單的線性迴歸和複雜的隨機森林表現差不多，我該選擇哪一個模型？為什麼？」

- **讓 AI 幫你修改這段程式的 Prompt（加入第三種模型）：**
  > 「請幫我在這段程式中加入第三個模型：『支援向量迴歸 (SVR)』或『Ridge 嶺迴歸』，並一起計算它的 5 折交叉驗證 R² 得分。」

---  
### 【步驟 5】：測試集正式評估與指標解讀 (MAE, RMSE, R²)
執行下方儲存格，在封存的「測試集 (X_test, y_test)」上進行最終大考，計算評估指標並繪製「真實值 vs. 預測值」散佈圖。

In [ ]:
# [步驟 5 程式碼] 測試集大考與指標計算
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 擬合訓練集並在測試集上推論
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

# 計算評估指標
eval_results = pd.DataFrame({
    "評估指標": ["MAE (平均絕對誤差)", "RMSE (均方根誤差)", "R² (判定係數)"],
    "多元線性迴歸數值": [
        f"{mean_absolute_error(y_test, y_pred_lr):.2f} 萬元",
        f"{root_mean_squared_error(y_test, y_pred_lr):.2f} 萬元",
        f"{r2_score(y_test, y_pred_lr):.4f}"
    ]
})
display(eval_results)

# 繪製真實值 vs 預測值散佈圖
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.5, color="royalblue", label="測試集樣本")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="完美預測線 (y=x)")
plt.title("測試集預測效果評估 (真實銷售額 vs. 模型預測額)", fontsize=13)
plt.xlabel("實際銷售額 (萬元)", fontsize=11)
plt.ylabel("模型預測銷售額 (萬元)", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們拿出從未參與訓練的測試集驗證。散佈圖上的所有預測點幾乎完美緊貼著紅色虛線（y=x），且 R² 為 0.999，平均誤差 MAE 只有 2.38 萬元，代表模型完全沒有過度擬合 (Overfitting)！

- **你可以複製問 AI 的 Prompt（理解指標）：**
  > 「請用最簡單的生活化例子向我解釋：MAE、RMSE 和 R² 分別代表什麼意思？如果老闆問我『這個模型預測到底準不準？誤差大概多少？』我該怎麼跟老闆報告？」

- **讓 AI 幫你修改這段程式的 Prompt（誤差分佈圖）：**
  > 「請幫我寫一段代碼，計算『殘差 (Residuals = 實際值 - 預測值)』，並繪製殘差的直方圖分佈 (Histogram)，檢查殘差是否符合常態分佈？」

---  
### 【步驟 6】：實務情境模擬預測 (Inference)
執行下方儲存格，建立一個方便重用的 `predict_sales` 函式。你可以輸入任意廣告預算組合，讓模型即時預估銷售總額！

In [ ]:
# [步驟 6 程式碼] 自訂預算情境試算函式
def predict_sales(pipeline, plan_dict):
    """輸入新廣告配置字典，自動透過 Pipeline 輸出預測銷售額"""
    input_df = pd.DataFrame([plan_dict])
    pred = pipeline.predict(input_df)[0]
    return pred

# 試算兩種不同的行銷投放提案
plan_A = {"電視廣告": 85.0, "廣播廣告": 25.0, "社群媒體廣告": 5.0, "網紅廣告": "大於百萬"}
plan_B = {"電視廣告": 40.0, "廣播廣告": 10.0, "社群媒體廣告": 2.0, "網紅廣告": "小於一萬"}

pred_A = predict_sales(model_lr, plan_A)
pred_B = predict_sales(model_lr, plan_B)

print("=== 商業廣告方案預測比較 ===")
print(f"提案 A (高預算加碼電視與大網紅) -> 預期銷售額: {pred_A:.2f} 萬元")
print(f"提案 B (低預算保守投放方案)     -> 預期銷售額: {pred_B:.2f} 萬元")
print(f"➡️ 提案 A 預計比 提案 B 多帶進 {pred_A - pred_B:.2f} 萬元的營收！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  模型訓練好的最大價值就是「輔助商業決策」。透過將 Pipeline 封裝成函式，我們只需傳入未經處理的原始字典，Pipeline 會自動幫我們完成編碼與縮放並輸出銷售額！

- **你可以複製問 AI 的 Prompt（商業決策提問）：**
  > 「如果我們公司下一季度的總行銷預算只有 100 千元，根據這個線性迴歸模型的係數權重，我該如何將這 100 千元分配給電視、廣播和社群廣告，才能達到最高的銷售效益？」

- **讓 AI 幫你修改這段程式的 Prompt（自訂你的提案）：**
  > 「請幫我寫一個互動式輸入（例如 Python 的 `input()` 或利用字典迴圈），讓我可以在 Notebook 中隨意輸入電視、廣播、社群費用與網紅規模，並即時印出預測銷售額。」